In [1]:
import os

In [2]:
%pwd

'd:\\Smart-Inventory-Defect-Detection-Quality-Control-API\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Smart-Inventory-Defect-Detection-Quality-Control-API'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    model_path: Path

In [6]:
from SIDD.constants import *
from SIDD.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=Path(config.root_dir),
            model_path=Path(config.model_path)
        )

        return model_evaluation_config

In [8]:
import json
from pathlib import Path

import torch

In [ ]:
import json
from pathlib import Path

import torch


class ModelEvaluation:
    def __init__(
        self,
        config: ModelEvaluationConfig,
        model,
        val_loader,
        criterion,
        dice_metric,
        iou_metric
    ):

        self.config = config
        self.model = model
        self.val_loader = val_loader
        self.criterion = criterion
        self.dice_metric = dice_metric
        self.iou_metric = iou_metric

        self.device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )

        self.model.to(self.device)

        checkpoint = torch.load(
            self.config.model_path,
            map_location=self.device
        )

        self.model.load_state_dict(checkpoint)

    def evaluate(self):

        self.model.eval()

        running_loss = 0.0
        running_dice = 0.0
        running_iou = 0.0

        num_batches = len(self.val_loader)

        with torch.no_grad():

            for images, masks in self.val_loader:

                images = images.to(self.device, non_blocking=True)
                masks = masks.to(self.device, non_blocking=True)

                with torch.amp.autocast("cuda"):

                    outputs = self.model(images)

                    loss = self.criterion(outputs, masks)

                preds = torch.sigmoid(outputs)
                preds = (preds > 0.5).float()

                dice = self.dice_metric(preds, masks)
                iou = self.iou_metric(preds, masks)

                running_loss += loss.item()
                running_dice += dice.item()
                running_iou += iou.item()

        results = {

            "loss": running_loss / num_batches,
            "dice": running_dice / num_batches,
            "iou": running_iou / num_batches

        }

        print("=" * 50)
        print("Evaluation Results")
        print("=" * 50)
        print(f"Loss : {results['loss']:.4f}")
        print(f"Dice : {results['dice']:.4f}")
        print(f"IoU  : {results['iou']:.4f}")
        print("=" * 50)

        output_path = Path(self.config.root_dir) / "metrics.json"

        with open(output_path, "w") as f:
            json.dump(results, f, indent=4)

        return results